# Thief Detector
## This task tests your Image Processing skills to build a motion detection algorithm that alarms you when you have an unwanted visitor in your home.

## Steps
- 1. Get the live video feed from your webcam
- 2. Fix a scene (the place you want to monitor) and store it as a reference background image
    - Store the first frame as the reference background frame
- 3. For every frame, check if there is any unwanted object inside the scene you are monitoring
    - Use **Background Subtraction** concept (**cv2.absdiff( )**)
        - Subtract the current frame from the reference background image(frame) to see the changes in the scene
        - If there is enormous amount of pixels distrubed in the subtraction result image
            - unwanted visitor (place is unsafe --> alarm the authorities)
        - If there is no enormous amount of pixels distrubed in the subtraction result image
            - no unwanted visitor (place is safe)
- 4. Output the text **"UNSAFE"** in **red** color on the top right of the frame when there is an intruder in the scene.
- 5. Save the live feed
- 6. Submit the (.ipynb) file

In [9]:
import cv2
import numpy as np

# Initialize webcam feed
cap = cv2.VideoCapture(0)

# Read the first frame and check if the feed is available
ret, frame = cap.read()
if not ret:
    print("Error: Couldn't read video feed.")
    cap.release()
    exit()

# Convert the first frame to grayscale and store it as the reference background
background = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# Apply Gaussian blur to reduce noise and improve motion detection
background = cv2.GaussianBlur(background, (21, 21), 0)  # Blurring for noise reduction

while True:
    # Capture a new frame from the webcam
    ret, frame = cap.read()
    if not ret:
        print("Error: Couldn't read video feed.")
        break  # Properly placed inside the loop to exit if the feed is interrupted
    
    # Convert the current frame to grayscale
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise and improve motion detection
    gray_frame = cv2.GaussianBlur(gray_frame, (21, 21), 0)
    
    # Compute the absolute difference between the current frame and the reference background
    frame_diff = cv2.absdiff(background, gray_frame)
    
    # Apply a threshold to detect changes (motion)
    _, thresh = cv2.threshold(frame_diff, 25, 255, cv2.THRESH_BINARY)
    
    # Find contours of the thresholded image
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Flag to check if there is an intruder
    intruder_detected = False
    
    # Iterate through contours
    for contour in contours:
        if cv2.contourArea(contour) > 500:  # Check if the contour area is large enough
            # Mark the area as an intruder
            intruder_detected = True
            (x, y, w, h) = cv2.boundingRect(contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)  # Draw rectangle around the intruder
    
    # If an intruder is detected, add "UNSAFE" text on the frame
    if intruder_detected:
        cv2.putText(frame, "UNSAFE", (frame.shape[1] - 150, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Display the frame with detection
    cv2.imshow("Thief Detector", frame)
    
    # Press 'q' to exit the loop and stop detection
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break  # Break the loop when 'q' is pressed

# Release the webcam and close any OpenCV windows
cap.release()
cv2.destroyAllWindows()


2025-03-28 21:18:26.680 python[41720:3572758] +[IMKClient subclass]: chose IMKClient_Modern
2025-03-28 21:18:26.680 python[41720:3572758] +[IMKInputSession subclass]: chose IMKInputSession_Modern
